# Sprint Block C+D — Clean, Filter & GPU Annotation

**Runtime for Block C (Cells 1–6): CPU.** **Switch to T4 GPU for Block D (Cell 7 onward)** — Runtime → Change runtime type → T4 GPU. The notebook detects the runtime and tells you when to switch.

Implements Hours 6–14 of the 48-hour runbook:
- **Block C**: markup strip → sentence split → language ID → MinHash near-dedup → two-gate relevance filter → `clean/corpus_v1.parquet`
- **Block D**: `adjective_extractor.py` with spaCy on GPU → entities, (adj, noun) pairs, tag hints, FP-Growth transactions → `annotated/` + `transactions/`

Same checkpoint discipline as Block A+B: every stage resumes from where it died. Prerequisite: Block A+B shards exist in `raw/`.

## Cell 1: Mount + config (same harness contract as Block A+B)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
ROOT = '/content/drive/MyDrive/geoproject'
os.environ['HF_HOME'] = f'{ROOT}/models_cache/hf'
os.environ['XDG_CACHE_HOME'] = f'{ROOT}/models_cache/xdg'

raw_shards = glob.glob(f'{ROOT}/raw/*_shard*.parquet')
print(f'Found {len(raw_shards)} raw shards')
assert raw_shards, 'No raw shards found — run Block A+B first!'

import torch
GPU = torch.cuda.is_available()
print('GPU available:', GPU, '—', torch.cuda.get_device_name(0) if GPU else 'CPU runtime (fine for Block C)')

## Cell 2: Installs (~3 min)

In [ ]:
%%capture
!pip install -q "datasketch>=1.6" "fasttext-wheel" "pandas>=2.0" "pyarrow>=14" "spacy>=3.7" tqdm
# fastText language-ID model (176 languages, ~130 MB) cached on Drive
import os
LID = '/content/drive/MyDrive/geoproject/models_cache/lid.176.bin'
if not os.path.exists(LID):
    os.system(f'wget -q -O {LID} https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin')

In [ ]:
!pip install -q fasttext

In [ ]:
!pip install -q numpy

In [ ]:
import inspect, textwrap
import fasttext.FastText as ft_module
from fasttext.FastText import _FastText

def patch_fasttext_numpy2():
    """fasttext-wheel calls np.array(probs, copy=False) in predict(), which
    NumPy >=2.0 rejects whenever a copy is actually needed. Colab's current
    image ships NumPy 2.x, so this breaks on a fresh runtime. Re-execs their
    exact predict() source with only that one line corrected to
    np.asarray(probs) -- the fix NumPy's own error message recommends --
    rather than downgrading numpy and risking a torch/spacy ABI conflict."""
    src = inspect.getsource(_FastText.predict)
    fixed_src = textwrap.dedent(src.replace('np.array(probs, copy=False)', 'np.asarray(probs)'))
    namespace = dict(vars(ft_module))
    exec(compile(fixed_src, '<patched fasttext predict>', 'exec'), namespace)
    _FastText.predict = namespace['predict']
    print('fastText predict() patched for numpy>=2 compatibility.')

patch_fasttext_numpy2()

import fasttext, pandas as pd
lid_model = fasttext.load_model(LID)
print('fastText LID loaded. Sanity:', lid_model.predict('The old fort overlooks the valley', k=1)[0][0])

In [ ]:
import fasttext, pandas as pd
lid_model = fasttext.load_model(LID)
print('fastText LID loaded. Sanity:', lid_model.predict('The old fort overlooks the valley', k=1)[0][0])

## Cell 3: Load raw shards + mechanical cleaning

Strips residual HTML/wiki artifacts, normalizes whitespace/quotes, splits multi-sentence records (Wikivoyage `content` fields often hold 2–4 sentences — each becomes its own record inheriting the anchor), drops fragments < 4 tokens.

In [ ]:
import re, hashlib
from tqdm.auto import tqdm

# Resume checkpoint: if cleaning already ran this session-series, skip it.
STAGE = f'{ROOT}/clean/_stage_sentences.parquet'
if os.path.exists(STAGE):
    sent_df = pd.read_parquet(STAGE)
    print('Resumed cleaned sentences from checkpoint:', len(sent_df))
else:
    df = pd.concat([pd.read_parquet(s) for s in raw_shards], ignore_index=True)
    print('Raw records:', len(df))

    TAG_RE   = re.compile(r'<[^>]+>')
    WIKI_RE  = re.compile(r"\[\[|\]\]|\{\{|\}\}|''+|^[*#:;]+", re.M)
    URL_RE   = re.compile(r'https?://\S+|www\.\S+')
    SPACE_RE = re.compile(r'\s+')
    SENT_RE  = r'(?<=[.!?])\s+(?=[A-Z؀-ۿ])'

    # Vectorized cleaning on the whole column — 10-50x faster than a Python
    # row loop. Order matters: strip URLs/HTML/wiki markup BEFORE collapsing
    # whitespace, so WIKI_RE's line-anchored ^[*#:;] rule (re.M) still fires.
    s = df['text'].astype(str)
    s = s.str.replace(URL_RE, '', regex=True)
    s = s.str.replace(TAG_RE, ' ', regex=True)
    s = s.str.replace(WIKI_RE, ' ', regex=True)
    s = (s.str.replace('’', "'").str.replace('‘', "'")
          .str.replace('“', '"').str.replace('”', '"'))
    df['text'] = s.str.replace(SPACE_RE, ' ', regex=True).str.strip()

    # Vectorized sentence split: one row per sentence, anchor fields inherited.
    df['text'] = df['text'].str.split(SENT_RE, regex=True)
    sent_df = df.explode('text', ignore_index=True)
    sent_df['text'] = sent_df['text'].str.strip()
    sent_df = sent_df[sent_df['text'].str.split().str.len() >= 4]

    sent_df['sent_id'] = [hashlib.sha1(t.lower().encode()).hexdigest()[:16]
                          for t in sent_df.text]
    sent_df = sent_df.drop_duplicates(subset='sent_id').reset_index(drop=True)
    sent_df.to_parquet(STAGE, index=False)   # checkpoint: re-run skips cleaning
print('Sentence records after split + exact dedup:', len(sent_df))

## Cell 4: Language routing

`en` → main pipeline. `ur` + Roman-Urdu (lexicon heuristic on top of fastText, which mislabels Roman Urdu) → **held bucket saved to Drive**, not discarded — that's your phase-2 dataset.

In [ ]:
RU_WORDS = {'hai','hain','ka','ki','ke','mein','main','aur','wala','wali',
            'nahi','nahin','bohat','bahut','acha','achi','yahan','wahan',
            'kya','kia','se','ko','par','bhi','tha','thi','ho','hota','hoti'}

# Resume checkpoint: language routing over 100k+ sentences is not free.
EN_STAGE = f'{ROOT}/clean/_stage_english.parquet'
if os.path.exists(EN_STAGE):
    sent_df = pd.read_parquet(EN_STAGE)
    print('Resumed language-routed English set from checkpoint:', len(sent_df))
else:
    def route_lang(text):
        label, conf = lid_model.predict(text.replace('\n', ' '), k=1)
        lang = label[0].replace('__label__', '')
        toks = set(re.findall(r'[a-z]+', text.lower()))
        if len(toks & RU_WORDS) >= 2:      # Roman-Urdu override
            return 'ur-roman'
        return lang

    tqdm.pandas(desc='language ID')
    sent_df['lang_detected'] = sent_df.text.progress_apply(route_lang)
    print(sent_df.lang_detected.value_counts().head(8))

    held = sent_df[sent_df.lang_detected.isin(['ur', 'ur-roman'])]
    held.to_parquet(f'{ROOT}/clean/held_urdu_bucket.parquet', index=False)
    sent_df = sent_df[sent_df.lang_detected == 'en'].reset_index(drop=True)
    sent_df.to_parquet(EN_STAGE, index=False)
    print(f'English kept: {len(sent_df)} | Urdu/Roman-Urdu held for v2: {len(held)}')

## Cell 5: MinHash near-duplicate removal

Wikivoyage prose overlaps listing content; Commons captions repeat across photo series. LSH at Jaccard ≥ 0.8 on word 3-shingles; keeps the copy with the best anchor quality (`listing` > `element` > `page`).

In [ ]:
from datasketch import MinHash, MinHashLSH

# Resume checkpoint: the LSH pass is the slowest CPU stage in Block C.
DEDUP_STAGE = f'{ROOT}/clean/_stage_deduped.parquet'
if os.path.exists(DEDUP_STAGE):
    sent_df = pd.read_parquet(DEDUP_STAGE)
    print('Resumed near-deduped set from checkpoint:', len(sent_df))
else:
    QUALITY_RANK = {'listing': 0, 'element': 1, 'page': 2}
    sent_df['q_rank'] = sent_df.anchor_quality.map(QUALITY_RANK).fillna(3)
    sent_df = sent_df.sort_values('q_rank').reset_index(drop=True)  # best anchor survives

    def shingles(t, n=3):
        w = t.lower().split()
        return {' '.join(w[i:i+n]) for i in range(max(1, len(w)-n+1))}

    lsh = MinHashLSH(threshold=0.8, num_perm=64)
    keep = []
    for r in tqdm(sent_df.itertuples(), total=len(sent_df), desc='MinHash LSH'):
        m = MinHash(num_perm=64)
        m.update_batch([s.encode() for s in shingles(r.text)])  # batched > per-shingle loop
        if not lsh.query(m):          # no near-dupe seen yet -> keep
            lsh.insert(r.sent_id, m)
            keep.append(r.Index)
    sent_df = sent_df.loc[keep].drop(columns='q_rank').reset_index(drop=True)
    sent_df.to_parquet(DEDUP_STAGE, index=False)
print('After near-dedup:', len(sent_df))

## Cell 6: Two-gate relevance filter → corpus_v1

Gate 1: contains a place-category noun OR a known toponym (place_name field or gazetteer hit). Gate 2: contains ≥ 1 adjective-ish token (fast suffix/lexicon heuristic — the real POS tagging happens in Block D; this gate just keeps FP-Growth's transaction density up). Failures go to `context_pool.parquet`, not the trash.

In [ ]:
PLACE_NOUNS = {
 'bazaar','market','fort','castle','mosque','church','temple','shrine',
 'valley','hill','mountain','lake','river','canal','park','garden','street',
 'road','bridge','museum','hotel','restaurant','cafe','shop','station',
 'city','town','village','district','beach','plateau','plain','tower',
 'palace','tomb','monument','gate','square','harbour','harbor','port',
 'island','forest','desert','glacier','pass','peak','dam','stadium',
 'university','school','hospital','library','zoo','mall','bakery','house',
 'building','area','place','spot','neighborhood','neighbourhood','quarter'}

COMMON_ADJ = {
 'old','new','ancient','historic','modern','large','small','big','narrow',
 'wide','busy','quiet','beautiful','scenic','famous','popular','local',
 'traditional','colonial','crowded','peaceful','sacred','grand','tall',
 'green','sprawling','bustling','charming','vibrant','cheap','expensive',
 'nice','great','good','best','stunning','picturesque','lively','serene',
 'magnificent','huge','tiny','lovely','pretty','quaint','remote','steep',
 'sacred','holy','ruined','abandoned','elegant','ornate','vast','pleasant'}
ADJ_SUFFIX = re.compile(r'\b\w+(?:ful|ous|ive|able|ible|al|ic|ish|esque|est)\b')

def gate(row):
    toks = set(re.findall(r'[a-z]+', row.text.lower()))
    # BUGFIX: a NaN place_name is truthy, so bool(row.place_name) let every
    # unnamed record pass gate 1. Guard with pd.notna + non-empty string.
    has_name = pd.notna(row.place_name) and bool(str(row.place_name).strip())
    g1 = bool(toks & PLACE_NOUNS) or has_name
    g2 = bool(toks & COMMON_ADJ) or bool(ADJ_SUFFIX.search(row.text.lower()))
    return g1 and g2

mask = [gate(r) for r in tqdm(sent_df.itertuples(), total=len(sent_df), desc='gates')]
corpus = sent_df[mask]
sent_df[[not m for m in mask]].to_parquet(f'{ROOT}/clean/context_pool.parquet', index=False)
corpus.to_parquet(f'{ROOT}/clean/corpus_v1.parquet', index=False)
print(f'corpus_v1: {len(corpus)} sentences | context pool: {(~pd.Series(mask)).sum()}')
print('\nBy source:'); print(corpus.source.value_counts())
print('\nBLOCK C DONE. Switch runtime to T4 GPU now (Runtime -> Change runtime type),')
print('then re-run Cells 1-2 and continue from Cell 7.')

# Block D — Annotation (T4 GPU from here)

## Cell 7: spaCy setup + adjective_extractor

Upload your `adjective_extractor.py` to `geoproject/` on Drive (one time). Model choice: `en_core_web_trf` on GPU (best parses), automatic fallback to `en_core_web_sm` on CPU per the runbook.

In [ ]:
%%capture
import torch
if torch.cuda.is_available():
    !pip install -q spacy[cuda12x] spacy-transformers
    !python -m spacy download en_core_web_trf -q
else:
    !python -m spacy download en_core_web_sm -q

In [ ]:
import os
print('File exists:', os.path.exists(f'{ROOT}/adjective_extractor.py'))
print('Contents of geoproject/:')
for f in sorted(os.listdir(ROOT)):
    print(' ', f)

In [ ]:
import importlib
importlib.invalidate_caches()

In [ ]:
import sys, spacy, torch
sys.path.insert(0, ROOT)               # adjective_extractor.py lives on Drive
import adjective_extractor as ax

if torch.cuda.is_available():
    spacy.require_gpu()
    MODEL = 'en_core_web_trf'
else:
    MODEL = 'en_core_web_sm'
    print('WARNING: no GPU — falling back to sm model (runbook fallback path)')

pi = ax.PlaceInterpreter(model=MODEL)
smoke = pi.interpret('The narrow winding streets of Raja Bazaar are lined with old brick shops.')
print('Model:', MODEL)
print('Smoke-test pairs:', smoke.pairs)
assert smoke.pairs, 'Extractor returned no pairs — check adjective_extractor.py upload'

## Cell 8: Batched GPU annotation with checkpointing

Uses `nlp.pipe` for GPU batching (the extractor's `interpret()` is refactored inline here to accept pre-parsed Docs — same logic, batched). Checkpoints every 2 000 sentences. 30 k sentences ≈ 20–40 min on T4; a few hours on CPU fallback.

In [ ]:
import json, sqlite3
from tqdm.auto import tqdm

corpus = pd.read_parquet(f'{ROOT}/clean/corpus_v1.parquet').reset_index(drop=True)

STATE = sqlite3.connect(f'{ROOT}/checkpoints/state.db')
STATE.execute("CREATE TABLE IF NOT EXISTS cursors (source TEXT PRIMARY KEY, pos TEXT)")
row = STATE.execute("SELECT pos FROM cursors WHERE source='annotate_v1'").fetchone()
start = int(row[0]) if row else 0
print(f'Annotating {len(corpus)} sentences, resuming at {start}')

BATCH, CKPT = 128, 2000
out_rows, shard_n = [], start // CKPT

def flush(rows, n):
    if rows:
        pd.DataFrame(rows).to_parquet(
            f'{ROOT}/annotated/annot_shard{n:05d}.parquet', index=False)

texts = corpus.text.tolist()[start:]
meta  = corpus.iloc[start:]

for i, doc in enumerate(tqdm(pi.nlp.pipe(texts, batch_size=BATCH),
                             total=len(texts), desc=MODEL)):
    # ---- inline of PlaceInterpreter.interpret(), operating on a done Doc ----
    entities, pairs, context, hints, basket = [], [], [], [], []
    for tok in doc:
        if tok.pos_ not in ('NOUN', 'PROPN') or tok.dep_ == 'compound':
            continue
        phrase, ptoks = pi._compound_phrase(tok)
        adjs = [a.lemma_.lower() for a in pi._adjectives_for(tok)]
        lemma = tok.lemma_.lower()
        if pi._is_named(tok):
            cat, key = 'named', None
        elif lemma in ax.OSM_NOUN_LEXICON:
            key = ax.OSM_NOUN_LEXICON[lemma][0]; cat = 'unnamed'
        else:
            cat, key = 'context', None; context.append(lemma)
        entities.append({'phrase': phrase, 'lemma': lemma,
                         'category': cat, 'osm_key': key, 'adjectives': adjs})
        for a in adjs:
            pairs.append((a, lemma, 'amod')); basket.append(f'amod:{a}_{lemma}')
            if a in ax.ADJ_DIRECT:
                k, v = ax.ADJ_DIRECT[a]; hints.append({'key': k, 'value': v})
            elif a in ax.ADJ_MODIFIER:
                k, v = ax.ADJ_MODIFIER[a]; hints.append({'key': k, 'value': v})
        for c in ptoks[:-1]:
            cl = c.lemma_.lower()
            if cl in ax.ADJ_DIRECT or cl in ax.ADJ_MODIFIER or c.pos_ == 'ADJ':
                pairs.append((cl, lemma, 'amod')); basket.append(f'amod:{cl}_{lemma}')
                if cl in ax.ADJ_DIRECT:
                    k, v = ax.ADJ_DIRECT[cl]; hints.append({'key': k, 'value': v})
                elif cl in ax.ADJ_MODIFIER:
                    k, v = ax.ADJ_MODIFIER[cl]; hints.append({'key': k, 'value': v})
            else:
                pairs.append((cl, lemma, 'compound')); basket.append(f'compound:{cl}_{lemma}')
        if lemma in ax.PLACE_HINT_NOUNS:
            basket.append('paraphrased_place')
    basket += [f'ctx:{c}' for c in context]
    basket += [f"type:{e['lemma']}" for e in entities if e['category'] == 'unnamed']
    m = meta.iloc[i]
    out_rows.append({'sent_id': m.sent_id, 'text': m.text, 'source': m.source,
                     'lat': m.lat, 'lon': m.lon,
                     'anchor_quality': m.anchor_quality,
                     'place_name': m.place_name,
                     'entities': json.dumps(entities),
                     'pairs': json.dumps(pairs),
                     'tag_hints': json.dumps(hints),
                     'transaction': json.dumps(basket),
                     'n_pairs': len(pairs)})
    if len(out_rows) >= CKPT:
        shard_n += 1; flush(out_rows, shard_n); out_rows = []
        STATE.execute("INSERT INTO cursors VALUES ('annotate_v1',?) "
                      "ON CONFLICT(source) DO UPDATE SET pos=excluded.pos",
                      (str(start + i + 1),)); STATE.commit()
shard_n += 1; flush(out_rows, shard_n)
STATE.execute("INSERT INTO cursors VALUES ('annotate_v1',?) "
              "ON CONFLICT(source) DO UPDATE SET pos=excluded.pos",
              (str(len(corpus)),)); STATE.commit()
print('Annotation complete.')

## Cell 9: Transaction export for FP-Growth (Block E's input) + stats

In [ ]:
annot = pd.concat([pd.read_parquet(s) for s in
                   sorted(glob.glob(f'{ROOT}/annotated/annot_shard*.parquet'))],
                  ignore_index=True).drop_duplicates(subset='sent_id')

# FP-Growth transaction file: one line per sentence, tab-separated items
with open(f'{ROOT}/transactions/transactions_v1.txt', 'w') as f:
    for t in annot.transaction:
        items = json.loads(t)
        if items:
            f.write('\t'.join(items) + '\n')

annot.to_parquet(f'{ROOT}/annotated/annotated_v1.parquet', index=False)

from collections import Counter
pair_counts = Counter()
for p in annot.pairs:
    for a, n, rel in json.loads(p):
        if rel == 'amod':
            pair_counts[f'{a} {n}'] += 1
print(f'Sentences annotated: {len(annot)}')
print(f'Mean adj-noun pairs/sentence: {annot.n_pairs.mean():.2f}  '
      f'(runbook target: >= 3 incl. compounds)')
print(f'Sentences with tag hints: {(annot.tag_hints != "[]").sum()}')
print('\nTop 15 adj-noun pairs (your Figure 1 preview):')
for pair, c in pair_counts.most_common(15):
    print(f'  {c:6d}  {pair}')
print('\nBLOCK C+D DONE -> transactions_v1.txt + annotated_v1.parquet ready for Block E (FP-Growth + ElasticNet).')

# Visuals from block a to d

In [ ]:
import os, json, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

ROOT = '/content/drive/MyDrive/geoproject'
FIGS = f'{ROOT}/figures_blockd'
os.makedirs(FIGS, exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 11})

print('Setup complete. Figures will save to:', FIGS)

In [ ]:
annot = pd.read_parquet(f'{ROOT}/annotated/annotated_v1.parquet')
print(f'Loaded {len(annot):,} annotated sentences')
print('\nColumns:', annot.columns.tolist())
print('\nFirst row (pairs and hints):'); print(json.loads(annot.iloc[0]['pairs'])[:3])
print(json.loads(annot.iloc[0]['tag_hints']))

In [ ]:
# Pipeline funnel — every count read from the parquet file that stage produced.
# Nothing hardcoded.
import pyarrow.parquet as pq

def _nrows(path, glob_pat=None):
    if glob_pat:
        return sum(pq.ParquetFile(p).metadata.num_rows for p in sorted(glob.glob(path)))
    return pq.ParquetFile(path).metadata.num_rows

stage_files = [
    ('Raw records\n(Block A+B)',        f'{ROOT}/raw/*_shard*.parquet',            True),
    ('Sentences\n(markup strip + split)', f'{ROOT}/clean/_stage_sentences.parquet', False),
    ('English only\n(language routing)',  f'{ROOT}/clean/_stage_english.parquet',   False),
    ('After near-dedup\n(MinHash LSH)',   f'{ROOT}/clean/_stage_deduped.parquet',   False),
    ('Passed relevance gate\n(corpus_v1)', f'{ROOT}/clean/corpus_v1.parquet',       False),
    ('Annotated\n(Block D)',             f'{ROOT}/annotated/annotated_v1.parquet',  False),
]
stages  = [s[0] for s in stage_files]
volumes = [_nrows(p, g) for _, p, g in stage_files]
for s, v in zip(stages, volumes):
    print(f'{s.replace(chr(10), " "):<38}: {v:,}')

colors = plt.cm.Blues(np.linspace(0.25, 0.95, len(stages)))
fig, ax = plt.subplots(figsize=(12, 5.5))
bars = ax.barh(range(len(stages)), volumes, color=colors, edgecolor='black', linewidth=1.3)
ax.set_yticks(range(len(stages))); ax.set_yticklabels(stages, fontsize=10)
ax.invert_yaxis()
for b, v in zip(bars, volumes):
    ax.text(v + max(volumes) * 0.012, b.get_y() + b.get_height() / 2, f'{v:,}',
            va='center', fontweight='bold', fontsize=10)
ax.set_xlabel('Sentence count', fontsize=12, fontweight='bold')
ax.set_title('Corpus construction pipeline (Blocks A–D)', fontsize=13, fontweight='bold')
ax.set_xlim(0, max(volumes) * 1.18)
ax.grid(axis='x', alpha=0.3)
# the count rises at step 2: multi-sentence source records are split into one row per sentence
ax.annotate('records split into\nindividual sentences', xy=(volumes[1], 1),
            xytext=(volumes[1] * 0.55, 2.1), fontsize=8, style='italic',
            arrowprops=dict(arrowstyle='->', lw=1))
plt.tight_layout()
plt.savefig(f'{FIGS}/01_pipeline.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIGS}/01_pipeline.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('saved: 01_pipeline')


In [ ]:
# Extract pair counts for each sentence
pair_counts = np.array([len(json.loads(p)) for p in annot['pairs']])

fig, ax = plt.subplots(figsize=(11, 6))
bins = np.arange(0, min(16, pair_counts.max() + 2)) - 0.5
counts, _, patches = ax.hist(pair_counts, bins=bins, edgecolor='black', color='#4ECDC4', linewidth=1.2, alpha=0.8)

# Add value labels
for patch, count in zip(patches, counts):
    height = patch.get_height()
    ax.text(patch.get_x() + patch.get_width()/2., height + 5000,
            f'{int(count):,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axvline(3, color='red', linestyle='--', linewidth=2.5, label='Target: ≥3 pairs', alpha=0.8)

ax.set_xlabel('Number of Adjective-Noun Pairs', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Sentences', fontsize=12, fontweight='bold')
ax.set_title(f'Distribution of Descriptive Pairs (Mean: {pair_counts.mean():.2f}, Median: {np.median(pair_counts):.0f})',
             fontsize=13, fontweight='bold')
ax.set_xticks(range(0, min(16, pair_counts.max() + 1)))
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)

pct_3plus = (pair_counts >= 3).sum() / len(pair_counts) * 100
ax.text(0.02, 0.95, f'{pct_3plus:.1f}% have ≥3 pairs', transform=ax.transAxes,
        fontsize=11, fontweight='bold', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3),
        verticalalignment='top')

plt.tight_layout()
plt.savefig(f'{FIGS}/02_pair_distribution.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIGS}/02_pair_distribution.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: 02_pair_distribution')

In [ ]:
# Extract all (adj, noun) pairs
pair_counter = Counter()
for pairs_json in annot['pairs']:
    pairs = json.loads(pairs_json)
    for adj, noun, rel in pairs:
        if rel == 'amod':  # only count actual adjective-noun relations
            pair_counter[f'{adj} {noun}'] += 1

top_20 = pair_counter.most_common(20)
pairs_list = [p[0] for p in top_20]
freqs = [p[1] for p in top_20]

fig, ax = plt.subplots(figsize=(12, 8))
colors_freq = plt.cm.viridis(np.linspace(0.3, 0.9, len(pairs_list)))
bars = ax.barh(range(len(pairs_list)), freqs, color=colors_freq, edgecolor='black', linewidth=0.8)

# Add frequency labels
for i, (bar, freq) in enumerate(zip(bars, freqs)):
    ax.text(freq + 30, bar.get_y() + bar.get_height()/2, f'{freq:,}',
            va='center', fontsize=10, fontweight='bold')

ax.set_yticks(range(len(pairs_list)))
ax.set_yticklabels(pairs_list, fontsize=11)
ax.set_xlabel('Frequency (occurrences)', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Adjective-Noun Patterns Extracted', fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGS}/03_top_patterns.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIGS}/03_top_patterns.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: 03_top_patterns')

In [ ]:
hint_counts = np.array([len(json.loads(h)) for h in annot['tag_hints']])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: pie chart
with_hints = (hint_counts > 0).sum()
without_hints = len(annot) - with_hints
sizes = [with_hints, without_hints]
labels_pie = [f'With Hints\n({with_hints:,})', f'No Hints\n({without_hints:,})']
colors_pie = ['#2ecc71', '#e74c3c']
wedges, texts, autotexts = ax1.pie(sizes, labels=labels_pie, autopct='%1.1f%%',
                                     colors=colors_pie, startangle=90, textprops={'fontsize': 11})
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax1.set_title('Sentences with DIRECT/MODIFIER Tag Hints', fontsize=12, fontweight='bold')

# Right: histogram of hint counts (for sentences with hints)
hints_with = hint_counts[hint_counts > 0]
ax2.hist(hints_with, bins=range(0, min(8, hints_with.max() + 2)), edgecolor='black', color='#2ecc71', alpha=0.8)
ax2.set_xlabel('Number of Tag Hints per Sentence', fontsize=11, fontweight='bold')
ax2.set_ylabel('Count', fontsize=11, fontweight='bold')
ax2.set_title(f'Distribution of Tag Hints (among {with_hints:,} sentences)', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGS}/04_tag_hints.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIGS}/04_tag_hints.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: 04_tag_hints')

In [ ]:
stats = {
    'Metric': [
        'Sentences Annotated',
        'Mean Pairs/Sentence',
        'Median Pairs/Sentence',
        'Max Pairs (single sentence)',
        'Sentences with ≥3 Pairs',
        'Sentences with Tag Hints',
        'Mean Tag Hints (when present)',
    ],
    'Value': [
        f'{len(annot):,}',
        f'{pair_counts.mean():.2f}',
        f'{np.median(pair_counts):.0f}',
        f'{pair_counts.max()}',
        f'{(pair_counts >= 3).sum():,} ({(pair_counts >= 3).sum()/len(annot)*100:.1f}%)',
        f'{(hint_counts > 0).sum():,} ({(hint_counts > 0).sum()/len(annot)*100:.1f}%)',
        f'{hints_with.mean():.2f}',
    ]
}

stats_df = pd.DataFrame(stats)

fig, ax = plt.subplots(figsize=(12, 5))
table = ax.table(cellText=stats_df.values, colLabels=stats_df.columns,
                 cellLoc='left', loc='center', colWidths=[0.6, 0.4])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

# Style header
for i in range(2):
    table[(0, i)].set_facecolor('#1f77b4')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(stats_df) + 1):
    for j in range(2):
        table[(i, j)].set_facecolor('#f0f0f0' if i % 2 == 0 else 'white')

ax.axis('off')
plt.title('Block D Annotation Statistics', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{FIGS}/05_stats_table.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIGS}/05_stats_table.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: 05_stats_table')

In [ ]:
# Source counts from your runs
source_counts = annot['source'].value_counts()
sources = source_counts.index.tolist()
counts = source_counts.values.tolist()
colors_src = plt.cm.Set3(np.linspace(0, 1, len(sources)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie
wedges, texts, autotexts = ax1.pie(counts, labels=sources, autopct='%1.1f%%',
                                     colors=colors_src, startangle=90, textprops={'fontsize': 11})
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax1.set_title('Annotated Records by Source', fontsize=12, fontweight='bold')

# Bar
bars = ax2.bar(sources, counts, color=colors_src, edgecolor='black', linewidth=1.2)
for bar, count in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2., count + 5000,
             f'{count:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('Count', fontsize=11, fontweight='bold')
ax2.set_title('Annotated Sentences by Source', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGS}/06_sources.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIGS}/06_sources.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: 06_sources')

In [ ]:
print("\n" + "=" * 70)
print("BLOCK D COMPLETION SUMMARY")
print("=" * 70)
print()
_top_pat, _top_n = pair_counter.most_common(1)[0]
print(f"Annotated sentences               : {len(annot):,}")
print(f"Mean adjective-noun pairs/sentence : {pair_counts.mean():.2f}  (target >= 3)")
print(f"Sentences with >= 3 pairs          : {(pair_counts >= 3).sum():,} "
      f"({(pair_counts >= 3).sum()/len(annot)*100:.1f}%)")
print(f"Sentences with tag hints           : {(hint_counts > 0).sum():,} "
      f"({(hint_counts > 0).sum()/len(annot)*100:.1f}%)")
print(f"Most frequent pattern              : '{_top_pat}' ({_top_n:,} occurrences)")
print()
print(f"Figures saved to: {FIGS}/")
print("Ready for Block E.")
print("=" * 70)
